In [1]:
import requests
from bs4 import BeautifulSoup
from astropy.io import fits
import matplotlib.pyplot as plt
from desispec.io.spectra import read_spectra
from desispec.coaddition import coadd_cameras
import numpy as np
from desitarget.targetmask import desi_mask
from astropy.table import Table
from desispec.resolution import Resolution
from scipy.optimize import curve_fit

In [ ]:
### Generates Lists of all the links needed to download the data
r_i = requests.get("https://data.desi.lbl.gov/public/dr1/spectro/redux/iron/healpix/main/dark/") 
soup_i = BeautifulSoup(r_i.content, "html.parser")
all_links_i = soup_i.find_all("a")
linkList_i = [link['href'] for link in all_links_i[1:]] # creates a list of all the first indicies of the coaad files

all_lists = []
for i in range(len(linkList_i)):
    r_j = requests.get("https://data.desi.lbl.gov/public/dr1/spectro/redux/iron/healpix/main/dark/" + str(linkList_i[i]))
    soup_j = BeautifulSoup(r_j.content, "html.parser")
    all_links_j = soup_j.find_all("a")
    linkList_j = [link['href'] for link in all_links_j] 
    all_lists.append(linkList_j[1:]) # creates a list of all the second indicies of the coaad files for each of the first indicies
    

In [12]:
### Initializing
tile = set()
zArray = []
RaArray = []
DecArray = []
tidArray = []
hdul = fits.open("qso_cat_dr1_main_dark_healpix_zlya-v0.fits") # lya quasar catalog
data = hdul[1].data
valid_logical = (data["Z"][:] < 2.5) & (data["ZWARN"][:] == 0) & (data["BI_CIV"][:] == 0)
zlya_tids = data["TARGETID"][valid_logical] # target ids of the valid quasars in the catalog


def gauss(x,sig,A,mu): # gaussian function
    return A*np.exp(-((x-mu)**2) / (2*sig**2))

In [ ]:
### Dowload and extract data

for i in range(len(linkList_i)):
    for j in range(len(all_lists[i])):
        if len(zArray) > 20000: # set to desired number of QSOs
            print("Reached desired number of QSOs")
            break
        else:
            if int(all_lists[i][j].replace("/","")) in tile: # checks if tile has already been downloaded and all spectra extracted
                print('coadd-main-dark-'+  str(all_lists[i][j]).replace("/","") + '.fits' + ' has already been downloaded')
                continue
            else:
                downFile = ('https://data.desi.lbl.gov/public/dr1/spectro/redux/iron/healpix/main/dark/' +  str(linkList_i[i]) # uses url generated from linkList and all_lists
                            + str(all_lists[i][j]) + 'coadd-main-dark-'+  str(all_lists[i][j]).replace("/","") + '.fits')
                !wget "{downFile}"
                print(all_lists[i][j].replace("/",""))

                spec = read_spectra("coadd-main-dark-" + all_lists[i][j].replace("/","") + ".fits" ) # gets spectra data from the coaad file
                fibermap = spec.fibermap
                coadd_dict = {tid: idx for idx, tid in enumerate(fibermap["TARGETID"])} # dictionary to map target ids to their index in the coadd file
                coadd_set = set(coadd_dict) # dictionary to map target ids to their index in the coadd file
                mask = np.fromiter((tid in coadd_set for tid in zlya_tids), dtype=bool)
                matching_tids = zlya_tids[mask]
                matching_tids_str = [str(tid) for tid in matching_tids] # target ids of the valid quasars that have spectra in the coadd file
                tidArray.extend(matching_tids_str) # adds the target ids of the valid quasars that have spectra in the coadd file to the tid array
                zArray.extend(data["Z"][mask]) # adds the redshifts of the valid quasars that have spectra in the coadd file to the z array
                RaArray.extend(data["TARGET_RA"][mask]) # adds the RA of the valid quasars that have spectra in the coadd file to the RA array
                DecArray.extend(data["TARGET_DEC"][mask])
                 # adds the DEC of the valid quasars that have spectra in the coadd file to the DEC array
                for tid in matching_tids: # loops through the target ids of the valid quasars that have spectra in the coadd file
                    spec_one = spec[coadd_dict[tid]]
                    wave = coadd_cameras(spec_one).wave['brz'] # extracts and combines the three arms for wave, flux, and ivar
                    flux = coadd_cameras(spec_one).flux['brz'][0]
                    ivar = coadd_cameras(spec_one).ivar['brz'][0]
                    pixel_mask = coadd_cameras(spec_one).mask['brz'][0] # saves array of masked/unmasked pixels

                    res = coadd_cameras(spec_one).resolution_data['brz'][0] # gets resolution data from current spectra
                    R  = Resolution(res) # creates sparse resolution matrix
                    sigArray = []
                    rArray = []
                    for jj in range(len(wave)): # generates
                        kernel = res[:, jj]  # shape: (ndiag,), centered at wavejj]
                        # Compute the pixel offsets relative to center
                        offsets = R.offsets
                        kernel_wave = wave[jj] + (offsets * np.gradient(wave)[jj])
                        if np.nan in kernel:
                            sigArray.append(np.nan)
                        elif np.inf in kernel:
                            sigArray.append(np.inf)
                        else:
                            # Get the resolution kernel (ISF) at pixel jj
                            kernel = res[:, jj]  # shape: (ndiag,), centered at wave[jj]
                            # Compute the pixel offsets relative to center
                            offsets = R.offsets
                            kernel_wave = wave[jj] + (offsets * np.gradient(wave)[jj])
                            peak = max(kernel)
                            try:
                                parameters = curve_fit(gauss, kernel_wave, kernel, p0=[0.5, 1, kernel_wave[5]], maxfev=10000)
                                sig_fit, A_fit, mu_fit = parameters[0]
                                sigArray.append(sig_fit)
                            except:
                                sigArray.append(np.nan)
                    t = Table()
                    t['WAVE'] = wave # formats the data into a table
                    t['FLUX'] = flux
                    t['IVAR'] = ivar
                    t['PIXEL_MASK'] = pixel_mask 
                    t['SIGMA_PIXEL'] = sigArray
                    t.write(str(tid) + '.fits',"overwrite=True") # writes the data to a fits file for analysis

                tile.add(int(all_lists[i][j].replace("/",""))) # used to make sure that if code stops while downloading, the last downloaded tile is saved so we dont have to restart
                filename = "coadd-main-dark-" + all_lists[i][j].replace("/","") + ".fits"
                !rm -rf "{filename}" # removes the downloaded coadd file
        
t2 = Table()
t2['TARGETID'] = tidArray 
t2['Z'] = zArray
t2['TARGET_RA'] = RaArray
t2['TARGET_DEC'] = DecArray
t2.write('QSO_catalog.fits',"overwrite=True") # writes a file with tid and redshifts of all the valid QSOs
